# 🏎️ JetBot 期末專案多功能控制面板 (分測 + 合併最終版)

本筆記本將整合程式分為三個部分，供您在實車落地時進行分步測試與最終合併演示：
- **單元一：純道路跟隨測試 (測試 PID 與轉向)** — 僅載入 ResNet 模型，不佔用 YOLO 與 PyCUDA 資源，適合調校 P/D Gain。
- **單元二：純交通路牌辨識測試 (測試 YOLO 與動作)** — 僅載入 YOLO 模型，測試路標停/走動作是否正常。
- **單元三：合併最終版 (道路跟隨 + 路牌辨識)** — 雙模型並行推論，進行期末考完整賽道演示。

> ⚠️ **重要守則**：每當您要從一個單元切換到另一個單元時，**必須執行該單元最下方的「安全停機與釋放相機」單元格**，否則會出現相機被佔用 (Resource busy) 的錯誤！

---

In [ ]:
## 🛠️ 零、道路跟隨 TensorRT 模型優化轉換 (選用)
# 0-1. 執行 TensorRT 編譯與轉換
import os
import torch
import torchvision.models as models

# 定位 JetBot 專案根目錄，避免 Jupyter 啟動目錄不同造成相對路徑失效
project_candidates = [
    os.getcwd(),
    os.path.expanduser('~/jetbot/notebooks/road_following_team_1'),
    '/home/nvidia/jetbot/notebooks/road_following_team_1'
]
PROJECT_ROOT = next((p for p in project_candidates
                     if os.path.isfile(os.path.join(p, 'Final_team_1.ipynb'))), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('找不到 road_following_team_1 專案根目錄')
PROJECT_ROOT = os.path.abspath(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f'專案根目錄：{PROJECT_ROOT}')

PYTORCH_MODEL = 'road_following_model/best_steering_model_xy.pth'
TRT_MODEL = 'road_following_model/best_steering_model_xy_trt.pth'
ONNX_MODEL = 'road_following_model/best_steering_model_xy.onnx'
ENGINE_MODEL = 'road_following_model/best_steering_model_xy.engine'

if not os.path.exists(PYTORCH_MODEL):
    print(f"❌ 錯誤：找不到 PyTorch 原始權重檔案：{PYTORCH_MODEL}")
    print("請確認您已將訓練好的 .pth 檔案上傳至對應目錄。")
else:
    print(f"正在載入 PyTorch 權重：{PYTORCH_MODEL}...")
    # 建立 ResNet-18 架構
    model = models.resnet18(pretrained=False)
    model.fc = torch.nn.Linear(512, 2)
    model.load_state_dict(torch.load(PYTORCH_MODEL, map_location='cuda'))
    
    # 搬移至 GPU 並設定為評估模式
    model = model.cuda().eval()
    
    # 1. 測試 PyTorch 運作是否正常
    print("正在進行 PyTorch 模型測試推論...")
    try:
        test_in = torch.zeros((1, 3, 224, 224)).cuda()
        with torch.no_grad():
            test_out = model(test_in)
        print(f"✓ PyTorch 測試推論成功！輸出值：{test_out.cpu().detach().numpy()}")
    except Exception as e:
        print(f"❌ 錯誤：PyTorch 測試推論失敗！原因：{e}")
        
    # 2. 匯出為 ONNX 格式
    print("正在將 PyTorch 模型匯出為 ONNX 格式...")
    try:
        sample_data = torch.zeros((1, 3, 224, 224)).cuda()
        torch.onnx.export(
            model,
            sample_data,
            ONNX_MODEL,
            input_names=["input_0"],
            output_names=["output_0"],
            opset_version=11,
            do_constant_folding=True
        )
        print(f"✓ ONNX 匯出成功！已儲存至：{ONNX_MODEL}")
        has_onnx = True
    except Exception as e:
        print(f"❌ 錯誤：ONNX 匯出失敗：{e}")
        has_onnx = False


    # 2-1. 釋放 PyTorch 模型以確保 trtexec 有足夠的 VRAM
    try:
        del model
        import gc
        gc.collect()
        torch.cuda.empty_cache()
        print("✓ 已釋放 PyTorch 模型以清空 VRAM")
    except Exception as e:
        print(f"釋放 VRAM 失敗：{e}")

    # 3. 使用 Nvidia 官方 trtexec 編譯器將 ONNX 編譯為 TensorRT .engine
    if has_onnx and os.path.exists(ONNX_MODEL):
        print("\n正在將 ONNX 編譯為 TensorRT 引擎 (這在 JetBot 上需要 1~2 分鐘，請耐心等候)...\n")
        print("提示：trtexec 會在獨立行程中執行，大幅降低 VRAM 記憶體不足 (OOM) 的機率。")
        
        # 尋找 trtexec 路徑
        trtexec_path = "/usr/src/tensorrt/bin/trtexec"
        if not os.path.exists(trtexec_path):
            trtexec_path = "trtexec"
            
        import subprocess
        
        # 嘗試 FP16 半精度模式編譯
        print("嘗試：FP16 半精度模式編譯...")
        cmd_fp16 = f"{trtexec_path} --onnx={ONNX_MODEL} --saveEngine={ENGINE_MODEL} --fp16 --workspace=1024"
        print(f"執行：{cmd_fp16}")
        
        res = subprocess.run(cmd_fp16, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        
        if res.returncode != 0:
            print("⚠️ 警告：FP16 編譯失敗，正在嘗試自動降級至 FP32 單精度模式編譯...")
            cmd_fp32 = f"{trtexec_path} --onnx={ONNX_MODEL} --saveEngine={ENGINE_MODEL} --workspace=1024"
            print(f"執行：{cmd_fp32}")
            res = subprocess.run(cmd_fp32, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            
        if res.returncode == 0:
            print("✓ TensorRT 引擎編譯成功！")

            # 重新載入 PyTorch 原始模型以進行驗證
            print("重新載入 PyTorch 模型進行輸出一致性驗證...")
            model = models.resnet18(pretrained=False)
            model.fc = torch.nn.Linear(512, 2)
            model.load_state_dict(torch.load(PYTORCH_MODEL, map_location='cuda'))
            model = model.cuda().eval()
            
            # 4. 讀取編譯好的引擎位元組，包裝成 torch2trt 專用的 TRTModule state_dict 格式
            print(f"正在將 .engine 包裝為 PyTorch 相容的 {TRT_MODEL}...")
            try:
                with open(ENGINE_MODEL, 'rb') as f:
                    engine_bytes = f.read()
                
                state_dict = {
                    "engine": bytearray(engine_bytes),
                    "input_names": ["input_0"],
                    "output_names": ["output_0"]
                }
                
                os.makedirs(os.path.dirname(TRT_MODEL), exist_ok=True)
                trt_model_tmp = TRT_MODEL + '.tmp'
                torch.save(state_dict, trt_model_tmp)

                # 5. 先驗證暫存檔，成功後才原子取代正式模型，避免留下無效 TRT 檔
                from torch2trt import TRTModule
                verify_trt = TRTModule()
                verify_trt.load_state_dict(torch.load(trt_model_tmp))
                with torch.no_grad():
                    pytorch_verify = model(sample_data).float()
                    trt_verify = verify_trt(sample_data).float()
                max_diff = torch.max(torch.abs(pytorch_verify - trt_verify)).item()
                if not torch.allclose(pytorch_verify, trt_verify, rtol=1e-2, atol=1e-2):
                    raise RuntimeError(f"PyTorch/TRT 輸出差異過大，max_diff={max_diff:.6f}")
                os.replace(trt_model_tmp, TRT_MODEL)
                print(f"✓ 轉換、儲存與一致性驗證成功！max_diff={max_diff:.6f}")
            except Exception as e:
                if 'trt_model_tmp' in locals() and os.path.exists(trt_model_tmp):
                    os.remove(trt_model_tmp)
                print(f"❌ 錯誤：包裝與儲存 TRTModule 失敗：{e}")
        else:
            print("❌ 錯誤：TensorRT 引擎編譯失敗！詳細資訊：")
            print(res.stderr.decode('utf-8', errors='ignore'))
            print("\n請嘗試重啟 Jupyter Kernel 以釋放更多系統資源。")


---

## 🏁 單元一：純道路跟隨測試 (調校 PID 參數)
此部分只載入 Project 5 的道路循跡模型，適合微調馬達轉向與 PD 參數。

In [ ]:
# A1. 匯入基礎套件
import os
import time
import cv2
import numpy as np
import torch
import torchvision.transforms as transforms
from torch2trt import TRTModule
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg, Robot

# 定位 JetBot 專案根目錄，避免 Jupyter 啟動目錄不同造成相對路徑失效
project_candidates = [
    os.getcwd(),
    os.path.expanduser('~/jetbot/notebooks/road_following_team_1'),
    '/home/nvidia/jetbot/notebooks/road_following_team_1'
]
PROJECT_ROOT = next((p for p in project_candidates
                     if os.path.isfile(os.path.join(p, 'Final_team_1.ipynb'))), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('找不到 road_following_team_1 專案根目錄')
PROJECT_ROOT = os.path.abspath(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f"✓ 已切換至專案根目錄：{PROJECT_ROOT}")

# A2. 載入 ResNet-18 TensorRT 模型
device = torch.device('cuda')
model_road = TRTModule()
model_road.load_state_dict(torch.load('road_following_model/best_steering_model_xy_trt.pth'))



In [ ]:
# A3. 初始化相機 (224x224) 與 Robot
import time
# 檢查現有相機實例的解析度是否符合要求，若不符合則重新建立
if hasattr(Camera, '_instance') and Camera._instance is not None:
    if Camera._instance.width != 224 or Camera._instance.height != 224:
        print('⚠️ 偵測到相機解析度不符，正在重置相機單例...')
        try:
            Camera._instance.stop()
            time.sleep(1.0)
        except Exception as e:
            print(f'停止舊相機實例失敗: {e}')
        Camera._instance = None

camera_r = Camera.instance(width=224, height=224, fps=10)
camera_r.start()
robot_r = Robot()
print('相機 (224x224) 與馬達初始化完成 ✓')


In [ ]:
# A4. 定義影像處理與滑桿介面
mean_r = torch.tensor([0.485, 0.456, 0.406], device=device, dtype=torch.float32)
std_r = torch.tensor([0.229, 0.224, 0.225], device=device, dtype=torch.float32)

def preprocess_road_only(image):
    image = cv2.resize(image, (224, 224))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.transpose((2, 0, 1))
    image = torch.from_numpy(image).to(device=device, dtype=torch.float32)
    image = image / 255.0
    image = (image - mean_r[:, None, None]) / std_r[:, None, None]
    return image.unsqueeze(0)

# 建立控制滑桿
speed_slider_r = widgets.FloatSlider(min=0.0, max=0.6, step=0.01, value=0.15, description='Speed')
p_slider_r = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.10, description='P Gain')
d_slider_r = widgets.FloatSlider(min=0.0, max=1.0, step=0.005, value=0.02, description='D Gain')
bias_slider_r = widgets.FloatSlider(min=-0.3, max=0.3, step=0.01, value=0.00, description='Bias')

# 建立遙測數據顯示
x_text_r = widgets.Label(value="X: 0.000")
y_text_r = widgets.Label(value="Y: 0.000")
steer_text_r = widgets.Label(value="Steering: 0.000")
motor_text_r = widgets.Label(value="L: 0.00, R: 0.00")

# 即時影像 Widget
image_widget_r = widgets.Image(format='jpeg', width=300, height=300)

# 左右排版與顯示
controls_box_r = widgets.VBox([
    widgets.HTML("<b>⚙️ PID 參數調校控制台</b>"),
    speed_slider_r,
    p_slider_r,
    d_slider_r,
    bias_slider_r,
    widgets.HTML("<hr><b>📊 即時遙測數據</b>"),
    x_text_r,
    y_text_r,
    steer_text_r,
    motor_text_r
])

panel_r = widgets.HBox([image_widget_r, controls_box_r])
display(panel_r)

In [ ]:
# A5. 循跡控制迴圈
angle_last_r = 0.0
control_time_last_r = None
pd_initialized_r = False
safety_latched_r = False

def execute_road_only(change):
    global angle_last_r, control_time_last_r, pd_initialized_r, safety_latched_r
    if safety_latched_r:
        robot_r.stop()
        return
        
    try:
        image = change['new']
        if image is None:
            raise ValueError("相機尚未提供有效影格")
        
        # 影像前處理與推論
        road_tensor = preprocess_road_only(image)
        with torch.no_grad():
            output = model_road(road_tensor).detach().float().cpu().numpy().flatten()
        if output.size < 2:
            raise ValueError(f"道路模型輸出長度錯誤：{output.size}")
        x = float(output[0])
        y = float(output[1])
        if not np.isfinite([x, y]).all():
            raise ValueError(f"道路模型輸出不是有限值：x={x}, y={y}")
        if abs(x) > 1.5 or abs(y) > 1.5:
            raise ValueError(f"道路模型輸出超出合理範圍：x={x:.3f}, y={y:.3f}")
        
        # Project 5 JetBot 控制座標：將預測點轉成車體前向距離
        y_forward = max((0.5 - y) / 2.0, 0.05)
        angle = np.arctan2(x, y_forward)
        
        # D 項以 10 FPS 為調參基準，依實際幀間隔補償並限制極端抖動
        now = time.monotonic()
        dt = 0.1 if control_time_last_r is None else max(now - control_time_last_r, 0.02)
        derivative = 0.0 if not pd_initialized_r else (angle - angle_last_r) * min(0.1 / dt, 2.0)
        pid = angle * p_slider_r.value + derivative * d_slider_r.value
        angle_last_r = angle
        control_time_last_r = now
        pd_initialized_r = True
        steering = pid + bias_slider_r.value
        
        # 彎道自動降速 (依轉向量計算速度係數，急彎最低降至設定速度的 60%)
        curve_scale = max(1.0 - abs(steering), 0.6)
        base_speed = speed_slider_r.value * curve_scale
        
        left_motor = max(min(base_speed + steering, 1.0), 0.0)
        right_motor = max(min(base_speed - steering, 1.0), 0.0)

        # 平滑的馬達死區補償 (線性映射，保留轉向差速)
        MIN_MOTOR_SPEED = 0.12
        if left_motor > 0.01:
            left_motor = MIN_MOTOR_SPEED + (1.0 - MIN_MOTOR_SPEED) * left_motor
        if right_motor > 0.01:
            right_motor = MIN_MOTOR_SPEED + (1.0 - MIN_MOTOR_SPEED) * right_motor

        
        # 驅動馬達
        robot_r.left_motor.value = left_motor
        robot_r.right_motor.value = right_motor
        
        # 更新即時數據顯示
        x_text_r.value = f"X: {x:.3f}"
        y_text_r.value = f"Y: {y:.3f}"
        steer_text_r.value = f"Steering: {steering:.3f} (Scale: {curve_scale:.2f})"
        motor_text_r.value = f"L: {left_motor:.2f}, R: {right_motor:.2f}"
        
        # 繪製綠色路徑點
        display_img = np.copy(image)
        pixel_x = int(x * 112 + 112)
        pixel_y = int(y * 112 + 112)
        cv2.circle(display_img, (pixel_x, pixel_y), 8, (0, 255, 0), -1)
        image_widget_r.value = bgr8_to_jpeg(display_img)
    except Exception as e:
        print(f"❌ [Unit 1 Error] Exception in execute_road_only: {e}")
        robot_r.stop()
        safety_latched_r = True


In [ ]:
# A6. 啟動單元一 (道路跟隨)
robot_r.stop()
try:
    camera_r.unobserve(execute_road_only, names='value')
except Exception:
    pass
angle_last_r = 0.0
control_time_last_r = None
pd_initialized_r = False
safety_latched_r = False
frame_deadline = time.monotonic() + 3.0
while camera_r.value is None and time.monotonic() < frame_deadline:
    time.sleep(0.05)
if camera_r.value is None:
    raise RuntimeError("單元一相機在 3 秒內未提供影格，馬達維持停止")
execute_road_only({'new': camera_r.value})
if not safety_latched_r:
    camera_r.observe(execute_road_only, names='value')
else:
    print('⚠️ 警告：相機未能成功關閉，建議手動關閉或重啟 Kernel。')
    raise RuntimeError("單元一首次推論失敗，已保持馬達停止")
print("🚗 單元一：純道路跟隨自動駕駛啟動！")

In [ ]:
# A7. 安全停止單元一 (切換單元前必執行！)
import gc
try:
    camera_r.unobserve(execute_road_only, names='value')
except Exception as e:
    print(f"Unobserve camera_r failed: {e}")
time.sleep(1.5)
robot_r.stop()
pd_initialized_r = False
control_time_last_r = None
camera_stopped_r = False
try:
    camera_r.stop()
    camera_stopped_r = True
except Exception as e:
    print(f"Stop camera_r failed: {e}")

# 只有底層相機成功停止後才清除 singleton，避免建立第二個 GStreamer pipeline
if camera_stopped_r and hasattr(Camera, '_instance') and Camera._instance is camera_r:
    Camera._instance = None
globals().pop('model_road', None)
globals().pop('camera_r', None)
gc.collect()
torch.cuda.empty_cache()
if camera_stopped_r:
    print("單元一已停止，模型、相機與 GPU cache 已釋放 ✓")
else:
    print('⚠️ 警告：相機未能成功關閉，建議手動關閉或重啟 Kernel。')


---

## 🚥 單元二：純交通路牌辨識測試
此部分只載入 Project 6 的 YOLO 號誌偵測模型，測試自走車看到路牌後的停、走動作是否完全正確。在此模式下，自走車會直線行駛，只反應路牌動作。

In [ ]:
# ========================================================
# 從 road_following_team_1/trt_yolv4-tiny-master 載入 YOLO 工具與插件
# ========================================================
import sys
import os
import time
import cv2
import numpy as np
import torch
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg, Robot

project_candidates = [
    os.getcwd(),
    os.path.expanduser('~/jetbot/notebooks/road_following_team_1'),
    '/home/nvidia/jetbot/notebooks/road_following_team_1'
]
PROJECT_ROOT = next((p for p in project_candidates
                     if os.path.isfile(os.path.join(p, 'yolo', 'yolov4-tiny-416.trt'))), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('找不到 road_following_team_1/yolo/yolov4-tiny-416.trt')
PROJECT_ROOT = os.path.abspath(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
yolo_lib_candidates = [
    os.path.join(PROJECT_ROOT, 'trt_yolv4-tiny-master'),
    os.path.join(PROJECT_ROOT, 'trt_yolov4-tiny-master')
]
YOLO_LIB_PATH = next((p for p in yolo_lib_candidates
                      if os.path.isfile(os.path.join(p, 'utils', 'yolo.py'))), None)
if YOLO_LIB_PATH is None:
    raise FileNotFoundError('找不到 trt_yolv4-tiny-master/utils/yolo.py')
plugin_path = os.path.join(YOLO_LIB_PATH, 'plugins', 'libyolo_layer.so')
if not os.path.isfile(plugin_path):
    raise FileNotFoundError(f'找不到 YOLO plugin：{plugin_path}')
original_cwd = os.getcwd()
os.chdir(YOLO_LIB_PATH)
if YOLO_LIB_PATH not in sys.path:
    sys.path.insert(0, YOLO_LIB_PATH)
try:
    try:
        import pycuda.autoprimaryctx
    except ModuleNotFoundError:
        import pycuda.driver as cuda
        try:
            cuda.init()
        except Exception:
            pass
    from utils.yolo import TRT_YOLO
finally:
    os.chdir(original_cwd)
print(f'✓ YOLO 工具庫：{YOLO_LIB_PATH}')
print(f'✓ YOLO 引擎：{os.path.join(PROJECT_ROOT, "yolo", "yolov4-tiny-416.trt")}')


In [ ]:
# B2. 載入 YOLOv4-tiny 號誌偵測模型
import pycuda.driver as cuda
from utils.yolo import TRT_YOLO
try:
    cuda.init()
except Exception:
    pass
cuda_context_s = cuda.Device(0).retain_primary_context()
cuda_context_s.push()
try:
    model_sign_only = TRT_YOLO('yolov4-tiny-416', (416, 416), 4)
finally:
    cuda_context_s.pop()
print("單元二：YOLO 號誌模型載入成功 ✓")

In [ ]:
# B3. 初始化相機 (416x416) 與 Robot
import time
# 檢查現有相機實例的解析度是否符合要求，若不符合則重新建立
if hasattr(Camera, '_instance') and Camera._instance is not None:
    if Camera._instance.width != 416 or Camera._instance.height != 416:
        print('⚠️ 偵測到相機解析度不符，正在重置相機單例...')
        try:
            Camera._instance.stop()
            time.sleep(1.0)
        except Exception as e:
            print(f'停止舊相機實例失敗: {e}')
        Camera._instance = None

camera_s = Camera.instance(width=416, height=416, fps=10)
camera_s.start()
robot_s = Robot()
print('相機 (416x416) 與馬達初始化完成 ✓')


In [ ]:
# B4. 建立滑桿介面
speed_slider_s = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.18, description='Base Speed')
alert_width_slider_s = widgets.IntSlider(min=20, max=150, step=5, value=50, description='Alert Width')
image_widget_s = widgets.Image(format='jpeg', width=416, height=416)

display(image_widget_s, speed_slider_s, alert_width_slider_s)

In [ ]:
# B5. 路牌偵測控制迴圈
stop_cooldown_s = 0.0
rail_cooldown_s = 0.0
stop_until_s = 0.0  # 新增：非阻塞停車結束時間戳
pedestrian_until_s = 0.0
blocked_latched_s = False
safety_latched_s = False

# 用於追蹤連續偵測幀數的計數器 (0: blocked, 1: pedestrian, 2: rail, 3: stop)
sign_counts_s = {0: 0, 1: 0, 2: 0, 3: 0}
sign_priority_s = (0, 3, 2, 1)  # blocked > stop > rail > pedestrian

def execute_sign_only(change):
    global stop_cooldown_s, rail_cooldown_s, stop_until_s, pedestrian_until_s, blocked_latched_s, safety_latched_s, sign_counts_s
    if blocked_latched_s or safety_latched_s:
        robot_s.stop()
        return
        
    try:
        img = change['new']
        if img is None:
            raise ValueError("相機尚未提供有效影格")
        current_time = time.monotonic()
        
        # 停車期間仍持續執行 YOLO，讓 blocked 可以立即升級為永久停止
        is_timed_stop = current_time < stop_until_s
        if is_timed_stop:
            robot_s.stop()
            
        # 偵測路牌
        cuda_context_s.push()
        try:
            boxes, confs, clss = model_sign_only.detect(img, conf_th=0.6)
        finally:
            cuda_context_s.pop()
        
        detected_signs = []
        best_signs_by_class = {}
        for box, conf, cls in zip(boxes, confs, clss):
            cid = int(cls)
            if cid not in sign_counts_s:
                continue
            safe_box = tuple(int(v) for v in box)
            width = max(0, safe_box[2] - safe_box[0])
            sign = {"width": width, "class_id": cid, "box": safe_box, "confidence": float(conf)}
            detected_signs.append(sign)
            previous = best_signs_by_class.get(cid)
            if previous is None or width > previous["width"]:
                best_signs_by_class[cid] = sign
        
        # 定義各類別個別的框寬門檻 (3: stop=50, 2: rail=30, 1: pedestrian=50, 0: blocked=50)
        slider_val = alert_width_slider_s.value
        width_thresholds = {
            3: slider_val,        # stop
            2: max(20, slider_val - 20), # rail (比 stop 小約 20px, 預設 30px)
            1: slider_val,        # pedestrian
            0: slider_val         # blocked
        }
        
        # 每個類別每幀最多只累加一次，避免同一幀多個重複框被誤認成連續兩幀
        active_classes_this_frame = set()
        for cid, sign in best_signs_by_class.items():
            if cid == 3 and current_time < stop_cooldown_s:
                continue
            if cid == 2 and current_time < rail_cooldown_s:
                continue
            if sign["width"] >= width_thresholds[cid]:
                active_classes_this_frame.add(cid)

        for cid in sign_counts_s:
            sign_counts_s[cid] = sign_counts_s[cid] + 1 if cid in active_classes_this_frame else 0

        eligible_classes = [cid for cid in sign_priority_s
                            if cid in active_classes_this_frame and sign_counts_s[cid] >= 2]
        active_sign = best_signs_by_class[eligible_classes[0]] if eligible_classes else None

        # 一般定時停車不接受新的 stop/rail/pedestrian，但 blocked 仍可立即覆蓋
        if is_timed_stop and 0 not in eligible_classes:
            for cid in (1, 2, 3):
                sign_counts_s[cid] = 0
            display_img = np.copy(img)
            cv2.putText(display_img, f"ACTION: Stopping ({stop_until_s - current_time:.1f}s)", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
            image_widget_s.value = bgr8_to_jpeg(cv2.resize(display_img, (224, 224)))
            return
        if is_timed_stop and 0 in eligible_classes:
            active_sign = best_signs_by_class[0]
                
        speed_multiplier = 1.0
        current_action = "Driving Straight"
        
        if active_sign is not None:
            cid = active_sign["class_id"]
            
            if cid == 3:  # stop (停車再開) ➡ 原地停止 2 秒
                current_action = "ACTION: Stop (2s)"
                print("[SIGN] Detected STOP. Stopping for 2s...")
                robot_s.stop()
                stop_until_s = current_time + 2.0
                stop_cooldown_s = current_time + 12.0
                sign_counts_s = {0: 0, 1: 0, 2: 0, 3: 0}
                display_img = np.copy(img)
                cv2.putText(display_img, current_action, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
                image_widget_s.value = bgr8_to_jpeg(cv2.resize(display_img, (224, 224)))
                return
            elif cid == 2:  # rail (鐵路平交道) ➡ 原地停止 5 秒
                current_action = "ACTION: Railway (5s)"
                print("[SIGN] Detected RAILWAY. Stopping for 5s...")
                robot_s.stop()
                stop_until_s = current_time + 5.0
                rail_cooldown_s = current_time + 15.0
                sign_counts_s = {0: 0, 1: 0, 2: 0, 3: 0}
                display_img = np.copy(img)
                cv2.putText(display_img, current_action, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
                image_widget_s.value = bgr8_to_jpeg(cv2.resize(display_img, (224, 224)))
                return
            elif cid == 1:  # pedestrian (當心行人) ➡ 觸發 1 秒減速
                current_action = "ACTION: Pedestrian (Slow 0.7x)"
                pedestrian_until_s = current_time + 1.0
            elif cid == 0:  # blocked (道路封閉) ➡ 看到時停止
                current_action = "ACTION: Blocked! STOP"
                print("[SIGN] Detected BLOCKED. Stopping...")
                robot_s.stop()
                display_img = np.copy(img)
                for sign in detected_signs:
                    box = sign['box']
                    cid_draw = sign['class_id']
                    cv2.rectangle(display_img, (box[0], box[1]), (box[2], box[3]), (0, 0, 255), 2)
                    cv2.putText(display_img, f"ID:{cid_draw} C:{sign['confidence']:.2f} W:{sign['width']} F:{sign_counts_s[cid_draw]}", (box[0], max(15, box[1] - 5)), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
                cv2.putText(display_img, current_action, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
                image_widget_s.value = bgr8_to_jpeg(cv2.resize(display_img, (224, 224)))
                return
                
        # 檢查減速狀態是否在 1 秒維持期內
        if current_time < pedestrian_until_s:
            speed_multiplier = 0.7
            if active_sign is None:
                current_action = "ACTION: Pedestrian (Latching 0.7x)"
                
        # 驅動馬達直行 (不修正方向)
        base_speed = speed_slider_s.value * speed_multiplier
        robot_s.left_motor.value = base_speed
        robot_s.right_motor.value = base_speed
        
        # 標記繪圖
        display_img = np.copy(img)
        for sign in detected_signs:
            box = sign["box"]
            cid = sign["class_id"]
            cv2.rectangle(display_img, (box[0], box[1]), (box[2], box[3]), (0, 0, 255), 2)
            cv2.putText(display_img, f"ID:{cid} C:{sign['confidence']:.2f} W:{sign['width']} F:{sign_counts_s[cid]}", (box[0], max(15, box[1] - 5)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        cv2.putText(display_img, current_action, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
        image_widget_s.value = bgr8_to_jpeg(cv2.resize(display_img, (224, 224)))
    except Exception as e:
        print(f"❌ [Unit 2 Error] Exception in execute_sign_only: {e}")
        robot_s.stop()
        safety_latched_s = True


In [ ]:
# B6. 啟動單元二 (路牌辨識)
robot_s.stop()
try:
    camera_s.unobserve(execute_sign_only, names='value')
except Exception:
    pass
stop_cooldown_s = 0.0
rail_cooldown_s = 0.0
stop_until_s = 0.0
pedestrian_until_s = 0.0
blocked_latched_s = False
safety_latched_s = False
sign_counts_s = {0: 0, 1: 0, 2: 0, 3: 0}

frame_deadline = time.monotonic() + 3.0
while camera_s.value is None and time.monotonic() < frame_deadline:
    time.sleep(0.05)
if camera_s.value is None:
    raise RuntimeError("單元二相機在 3 秒內未提供影格，馬達維持停止")
execute_sign_only({'new': camera_s.value})
if not safety_latched_s and not blocked_latched_s:
    camera_s.observe(execute_sign_only, names='value')
elif safety_latched_s:
    raise RuntimeError("單元二首次推論失敗，已保持馬達停止")
print("🚗 單元二：純路牌辨識偵測啟動！")

In [ ]:
# B7. 安全停止單元二 (切換單元前必執行！)
import gc
try:
    camera_s.unobserve(execute_sign_only, names='value')
except Exception as e:
    print(f"Unobserve camera_s failed: {e}")
time.sleep(1.5)
robot_s.stop()
camera_stopped_s = False
try:
    camera_s.stop()
    camera_stopped_s = True
except Exception as e:
    print(f"Stop camera_s failed: {e}")

try:
    cuda.Context.synchronize()
except Exception as e:
    print(f"CUDA synchronize failed: {e}")
# 只有底層相機成功停止後才清除 singleton
if camera_stopped_s and hasattr(Camera, '_instance') and Camera._instance is camera_s:
    Camera._instance = None
globals().pop('model_sign_only', None)
globals().pop('cuda_context_s', None)
globals().pop('camera_s', None)
gc.collect()
torch.cuda.empty_cache()
if camera_stopped_s:
    print("單元二已停止，YOLO、相機與 GPU cache 已釋放 ✓")
else:
    print('⚠️ 警告：相機未能成功關閉，建議手動關閉或重啟 Kernel。')


---

## 🏁🏁 單元三：合併最終版 (道路跟隨 + 路牌辨識)
雙模型載入並行推論。用大圖 (416x416) 偵測路牌，偵測後即時下採樣 (224x224) 預測方向，配合 PD 運算驅動馬達。此部分為期末驗收使用的最終演示版本。

In [ ]:
# C1. 匯入完整套件
import os
import time
import cv2
import numpy as np
import torch
import torchvision.transforms as transforms
from torch2trt import TRTModule
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg, Robot

device = torch.device('cuda') # 確保定義 device

# 從 road_following_team_1/trt_yolv4-tiny-master 載入 YOLO 推論庫與插件
import sys
project_candidates = [
    os.getcwd(),
    os.path.expanduser('~/jetbot/notebooks/road_following_team_1'),
    '/home/nvidia/jetbot/notebooks/road_following_team_1'
]
PROJECT_ROOT = next((p for p in project_candidates
                     if os.path.isfile(os.path.join(p, 'yolo', 'yolov4-tiny-416.trt'))), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('找不到 road_following_team_1/yolo/yolov4-tiny-416.trt')
PROJECT_ROOT = os.path.abspath(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
yolo_lib_candidates = [
    os.path.join(PROJECT_ROOT, 'trt_yolv4-tiny-master'),
    os.path.join(PROJECT_ROOT, 'trt_yolov4-tiny-master')
]
YOLO_LIB_PATH = next((p for p in yolo_lib_candidates
                      if os.path.isfile(os.path.join(p, 'utils', 'yolo.py'))), None)
if YOLO_LIB_PATH is None:
    raise FileNotFoundError('找不到 trt_yolv4-tiny-master/utils/yolo.py')
plugin_path = os.path.join(YOLO_LIB_PATH, 'plugins', 'libyolo_layer.so')
if not os.path.isfile(plugin_path):
    raise FileNotFoundError(f'找不到 YOLO plugin：{plugin_path}')
original_cwd = os.getcwd()
os.chdir(YOLO_LIB_PATH)
if YOLO_LIB_PATH not in sys.path:
    sys.path.insert(0, YOLO_LIB_PATH)
try:
    try:
        import pycuda.autoprimaryctx
    except ModuleNotFoundError:
        import pycuda.driver as cuda
        try:
            cuda.init()
        except Exception:
            pass
    from utils.yolo import TRT_YOLO
finally:
    os.chdir(original_cwd)
print(f'✓ YOLO 工具庫：{YOLO_LIB_PATH}')
print(f'✓ YOLO 引擎：{os.path.join(PROJECT_ROOT, "yolo", "yolov4-tiny-416.trt")}')


In [ ]:
# C2. 載入雙模型 (道路跟隨 ResNet TRT + YOLOv4-tiny TRT)
import pycuda.driver as cuda
from utils.yolo import TRT_YOLO
try:
    cuda.init()
except Exception:
    pass
cuda_context_f = cuda.Device(0).retain_primary_context()
cuda_context_f.push()
try:
    model_road_final = TRTModule()
    model_road_final.load_state_dict(torch.load('road_following_model/best_steering_model_xy_trt.pth'))
    model_sign_final = TRT_YOLO('yolov4-tiny-416', (416, 416), 4)
finally:
    cuda_context_f.pop()
print("最終合併版：雙模型載入成功 ✓")

In [ ]:
# C3. 初始化相機 (224x224) 與 Robot
import time
# 檢查現有相機實例的解析度是否符合要求，若不符合則重新建立
if hasattr(Camera, '_instance') and Camera._instance is not None:
    if Camera._instance.width != 224 or Camera._instance.height != 224:
        print('⚠️ 偵測到相機解析度不符，正在重置相機單例...')
        try:
            Camera._instance.stop()
            time.sleep(1.0)
        except Exception as e:
            print(f'停止舊相機實例失敗: {e}')
        Camera._instance = None

camera_f = Camera.instance(width=224, height=224, fps=10)
camera_f.start()
robot_f = Robot()
print('最終合併版：相機 (224x224) 與馬達初始化完成 ✓')


In [ ]:
# C4. 正規化與滑桿介面
mean_f = torch.tensor([0.485, 0.456, 0.406], device=device, dtype=torch.float32)
std_f = torch.tensor([0.229, 0.224, 0.225], device=device, dtype=torch.float32)

def preprocess_road_final(image):
    image = cv2.resize(image, (224, 224))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.transpose((2, 0, 1))
    image = torch.from_numpy(image).to(device=device, dtype=torch.float32)
    image = image / 255.0
    image = (image - mean_f[:, None, None]) / std_f[:, None, None]
    return image.unsqueeze(0)

# 建立控制滑桿
speed_slider_f = widgets.FloatSlider(min=0.0, max=0.6, step=0.01, value=0.15, description='Speed')
p_slider_f = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.10, description='P Gain')
d_slider_f = widgets.FloatSlider(min=0.0, max=1.0, step=0.005, value=0.02, description='D Gain')
bias_slider_f = widgets.FloatSlider(min=-0.3, max=0.3, step=0.01, value=0.00, description='Bias')
alert_width_slider_f = widgets.IntSlider(min=20, max=150, step=5, value=50, description='Alert Width')

# 建立遙測數據顯示
x_text_f = widgets.Label(value="X: 0.000")
y_text_f = widgets.Label(value="Y: 0.000")
steer_text_f = widgets.Label(value="Steering: 0.000")
motor_text_f = widgets.Label(value="L: 0.00, R: 0.00")
action_text_f = widgets.Label(value="Action: Following Lane")

# 即時影像 Widget
image_widget_f = widgets.Image(format='jpeg', width=416, height=416)

# 左右排版與顯示
controls_box_f = widgets.VBox([
    widgets.HTML("<b>⚙️ 合併版自動駕駛控制台</b>"),
    speed_slider_f,
    p_slider_f,
    d_slider_f,
    bias_slider_f,
    alert_width_slider_f,
    widgets.HTML("<hr><b>📊 即時遙測數據</b>"),
    x_text_f,
    y_text_f,
    steer_text_f,
    motor_text_f,
    action_text_f
])

panel_f = widgets.HBox([image_widget_f, controls_box_f])
display(panel_f)

In [ ]:
# C5. 合併自動駕駛核心迴圈
angle_last_f = 0.0
control_time_last_f = None
pd_initialized_f = False
stop_cooldown_f = 0.0
rail_cooldown_f = 0.0
stop_until_f = 0.0
pedestrian_until_f = 0.0
blocked_latched_f = False
safety_latched_f = False

# 用於分頻推論的影格計數器與路牌快取
frame_count_f = 0
last_detected_signs_f = []
last_boxes_confs_clss_f = ([], [], [])
last_blocked_time_f = 0.0  # 新增：記錄上一次看到 blocked 路牌的時間戳

YOLO_INTERVAL = 2  # 每 2 幀執行一次 YOLO 偵測
SIGN_TRIGGER_THRESHOLD = 1  # 只需要連續 1 幀偵測到就觸發（適合高速行駛）

# 用於追蹤連續偵測幀數的計數器 (0: blocked, 1: pedestrian, 2: rail, 3: stop)
sign_counts_f = {0: 0, 1: 0, 2: 0, 3: 0}
sign_priority_f = (0, 3, 2, 1)  # blocked > stop > rail > pedestrian

def execute_final(change):
    global angle_last_f, control_time_last_f, pd_initialized_f, stop_cooldown_f, rail_cooldown_f, stop_until_f, pedestrian_until_f, blocked_latched_f, safety_latched_f, sign_counts_f
    global frame_count_f, last_detected_signs_f, last_boxes_confs_clss_f, last_blocked_time_f
    if blocked_latched_f or safety_latched_f:
        robot_f.stop()
        return
        
    try:
        img = change['new']
        if img is None:
            raise ValueError("相機尚未提供有效影格")
        current_time = time.monotonic()
        
        # 增加影格計數
        frame_count_f += 1
        
        # 檢查是否處於 blocked 停止狀態 (看到 blocked 或移除後未滿 1 秒)
        is_blocked_stop = (current_time - last_blocked_time_f < 1.0)
        
        # 停車期間仍持續執行 YOLO (每 YOLO_INTERVAL 幀一次)，讓 blocked 可以立即升級為永久停止
        is_timed_stop = current_time < stop_until_f
        
        if is_blocked_stop or is_timed_stop:
            robot_f.stop()
            pd_initialized_f = False
            control_time_last_f = None
            
        # 設定是否在這一幀執行 YOLO
        run_yolo = (frame_count_f % YOLO_INTERVAL == 0)
        
        if run_yolo:
            # 將 224x224 影像放大為 YOLO 需求的 416x416
            img_yolo = cv2.resize(img, (416, 416))
            
            # 僅在 YOLO 推論時推入 CUDA 上下文，結束後立即彈出
            cuda_context_f.push()
            try:
                boxes, confs, clss = model_sign_final.detect(img_yolo, conf_th=0.6)
            finally:
                cuda_context_f.pop()
                
            last_boxes_confs_clss_f = (boxes, confs, clss)
        else:
            boxes, confs, clss = last_boxes_confs_clss_f
            
        detected_signs = []
        best_signs_by_class = {}
        for box, conf, cls in zip(boxes, confs, clss):
            cid = int(cls)
            if cid not in sign_counts_f:
                continue
            safe_box = tuple(int(v) for v in box)
            width = max(0, safe_box[2] - safe_box[0])
            sign = {"width": width, "class_id": cid, "box": safe_box, "confidence": float(conf)}
            detected_signs.append(sign)
            previous = best_signs_by_class.get(cid)
            if previous is None or width > previous["width"]:
                best_signs_by_class[cid] = sign
                
        # 定義各類別個別的框寬門檻 (3: stop=50, 2: rail=30, 1: pedestrian=50, 0: blocked=50)
        slider_val = alert_width_slider_f.value
        width_thresholds = {
            3: slider_val,        # stop
            2: max(20, slider_val - 20), # rail (比 stop 小約 20px, 預設 30px)
            1: slider_val,        # pedestrian
            0: slider_val         # blocked
        }
        
        if run_yolo:
            # 每個類別每幀最多只累加一次，並依安全優先權選擇動作
            active_classes_this_frame = set()
            for cid, sign in best_signs_by_class.items():
                if cid == 3 and current_time < stop_cooldown_f:
                    continue
                if cid == 2 and current_time < rail_cooldown_f:
                    continue
                if sign["width"] >= width_thresholds[cid]:
                    active_classes_this_frame.add(cid)

            for cid in sign_counts_f:
                sign_counts_f[cid] = sign_counts_f[cid] + 1 if cid in active_classes_this_frame else 0
            last_detected_signs_f = detected_signs
        else:
            detected_signs = last_detected_signs_f

        # 取得滿足連續偵測次數的路牌
        eligible_classes = [cid for cid in sign_priority_f
                            if cid in best_signs_by_class and sign_counts_f[cid] >= SIGN_TRIGGER_THRESHOLD]
        active_sign = best_signs_by_class[eligible_classes[0]] if eligible_classes else None

        # 1. 處理 Blocked 停止狀態的提早返回
        if is_blocked_stop:
            for cid in (1, 2, 3):
                sign_counts_f[cid] = 0
            display_img = np.copy(img)
            # 繪製快取的路牌邊界框 (等比縮小至 224x224)
            for sign in detected_signs:
                box = sign["box"]
                cid_draw = sign["class_id"]
                if cid_draw == 0:
                    bx1, by1 = int(box[0]*224/416), int(box[1]*224/416)
                    bx2, by2 = int(box[2]*224/416), int(box[3]*224/416)
                    cv2.rectangle(display_img, (bx1, by1), (bx2, by2), (0, 0, 255), 2)
            
            remaining_time = max(0.0, 1.0 - (current_time - last_blocked_time_f))
            cv2.putText(display_img, f"ACTION: Blocked ({remaining_time:.1f}s)", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
            image_widget_f.value = bgr8_to_jpeg(display_img)
            action_text_f.value = "Action: Blocked Stop"
            
            # 若在此時 YOLO 再次偵測到 blocked，更新最後看到的時間戳
            if active_sign is not None and active_sign["class_id"] == 0:
                last_blocked_time_f = current_time
            return

        # 2. 處理一般定時停車的提早返回
        if is_timed_stop and 0 not in eligible_classes:
            for cid in (1, 2, 3):
                sign_counts_f[cid] = 0
            display_img = np.copy(img)
            cv2.putText(display_img, f"ACTION: Stopping ({stop_until_f - current_time:.1f}s)", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
            image_widget_f.value = bgr8_to_jpeg(display_img)
            action_text_f.value = "Action: Timed Stop"
            return
            
        if is_timed_stop and 0 in eligible_classes:
            active_sign = best_signs_by_class[0]

        speed_multiplier = 1.0
        current_action = "Following Lane"

        if active_sign is not None:
            cid = active_sign["class_id"]

            if cid == 3:  # stop (停車再開) ➡ 原地停止 2 秒
                current_action = "ACTION: Stop (2s)"
                print("[SIGN] Detected STOP. Stopping for 2s...")
                robot_f.stop()
                stop_until_f = current_time + 2.0
                stop_cooldown_f = current_time + 12.0
                sign_counts_f = {0: 0, 1: 0, 2: 0, 3: 0}
                pd_initialized_f = False
                control_time_last_f = None
                display_img = np.copy(img)
                cv2.putText(display_img, current_action, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
                image_widget_f.value = bgr8_to_jpeg(display_img)
                action_text_f.value = f"Action: {current_action}"
                return

            elif cid == 2:  # rail (鐵路平交道) ➡ 原地停止 5 秒
                current_action = "ACTION: Railway (5s)"
                print("[SIGN] Detected RAILWAY. Stopping for 5s...")
                robot_f.stop()
                stop_until_f = current_time + 5.0
                rail_cooldown_f = current_time + 15.0
                sign_counts_f = {0: 0, 1: 0, 2: 0, 3: 0}
                pd_initialized_f = False
                control_time_last_f = None
                display_img = np.copy(img)
                cv2.putText(display_img, current_action, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
                image_widget_f.value = bgr8_to_jpeg(display_img)
                action_text_f.value = f"Action: {current_action}"
                return

            elif cid == 1:  # pedestrian (當心行人) ➡ 減速 0.7 倍
                current_action = "ACTION: Pedestrian (Slow 0.7x)"
                pedestrian_until_f = current_time + 1.0

            elif cid == 0:  # blocked (道路封閉) ➡ 看到時停止，暫存時間戳
                current_action = "ACTION: Blocked! STOP"
                print("[SIGN] Detected BLOCKED. Stopping...")
                robot_f.stop()
                last_blocked_time_f = current_time
                pd_initialized_f = False
                control_time_last_f = None
                display_img = np.copy(img)
                for sign in detected_signs:
                    box = sign['box']
                    cid_draw = sign['class_id']
                    if cid_draw == 0:
                        bx1, by1 = int(box[0]*224/416), int(box[1]*224/416)
                        bx2, by2 = int(box[2]*224/416), int(box[3]*224/416)
                        cv2.rectangle(display_img, (bx1, by1), (bx2, by2), (0, 0, 255), 2)
                        cv2.putText(display_img, f"ID:{cid_draw} W:{sign['width']}", (bx1, max(15, by1 - 5)), 
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
                cv2.putText(display_img, current_action, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
                image_widget_f.value = bgr8_to_jpeg(display_img)
                action_text_f.value = f"Action: {current_action}"
                return

        if current_time < pedestrian_until_f:
            speed_multiplier = 0.7
            if active_sign is None:
                current_action = "ACTION: Pedestrian (Latching 0.7x)"

        # 3. 道路循跡推論 (img 解析度已是 224x224，直接送入)
        road_tensor = preprocess_road_final(img)
        with torch.no_grad():
            output = model_road_final(road_tensor).detach().float().cpu().numpy().flatten()
        if output.size < 2:
            raise ValueError(f"道路模型輸出長度錯誤：{output.size}")
        x = float(output[0])
        y = float(output[1])
        if not np.isfinite([x, y]).all():
            raise ValueError(f"道路模型輸出不是有限值：x={x}, y={y}")
        if abs(x) > 1.5 or abs(y) > 1.5:
            raise ValueError(f"道路模型輸出超出合理範圍：x={x:.3f}, y={y:.3f}")

        # Project 5 JetBot 控制座標：將預測點轉成車體前向距離
        y_forward = max((0.5 - y) / 2.0, 0.05)
        angle = np.arctan2(x, y_forward)

        # D 項以 10 FPS 為調參基準，依實際幀間隔補償並限制極端抖動
        now = time.monotonic()
        dt = 0.1 if control_time_last_f is None else max(now - control_time_last_f, 0.02)
        derivative = 0.0 if not pd_initialized_f else (angle - angle_last_f) * min(0.1 / dt, 2.0)
        pid = angle * p_slider_f.value + derivative * d_slider_f.value
        angle_last_f = angle
        control_time_last_f = now
        pd_initialized_f = True
        steering = pid + bias_slider_f.value

        # 彎道自動降速 (依轉向量計算速度係數，急彎最低降至設定速度的 60%)
        curve_scale = max(1.0 - abs(steering), 0.6)
        base_speed = speed_slider_f.value * speed_multiplier * curve_scale

        # 計算馬達出力
        left_motor = max(min(base_speed + steering, 1.0), 0.0)
        right_motor = max(min(base_speed - steering, 1.0), 0.0)

        # 平滑的馬達死區補償 (線性映射，保留轉向差速)
        MIN_MOTOR_SPEED = 0.12
        if left_motor > 0.01:
            left_motor = MIN_MOTOR_SPEED + (1.0 - MIN_MOTOR_SPEED) * left_motor
        if right_motor > 0.01:
            right_motor = MIN_MOTOR_SPEED + (1.0 - MIN_MOTOR_SPEED) * right_motor

        # 驅動馬達
        robot_f.left_motor.value = left_motor
        robot_f.right_motor.value = right_motor

        # 更新即時數據顯示
        x_text_f.value = f"X: {x:.3f}"
        y_text_f.value = f"Y: {y:.3f}"
        steer_text_f.value = f"Steering: {steering:.3f} (Curve Scale: {curve_scale:.2f})"
        motor_text_f.value = f"L: {left_motor:.2f}, R: {right_motor:.2f}"
        action_text_f.value = f"Action: {current_action}"

        # 4. 繪圖標記 (在 224x224 畫面上繪圖)
        display_img = np.copy(img)
        pixel_x = int(x * 112 + 112)
        pixel_y = int(y * 112 + 112)
        cv2.circle(display_img, (pixel_x, pixel_y), 8, (0, 255, 0), -1)

        for sign in detected_signs:
            box = sign["box"]
            cid = sign["class_id"]
            bx1, by1 = int(box[0]*224/416), int(box[1]*224/416)
            bx2, by2 = int(box[2]*224/416), int(box[3]*224/416)
            cv2.rectangle(display_img, (bx1, by1), (bx2, by2), (0, 0, 255), 2)
            cv2.putText(display_img, f"ID:{cid} W:{sign['width']}", (bx1, max(15, by1 - 5)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

        cv2.putText(display_img, current_action, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
        image_widget_f.value = bgr8_to_jpeg(display_img)
    except Exception as e:
        print(f"❌ [Unit 3 Error] Exception in execute_final: {e}")
        robot_f.stop()
        safety_latched_f = True


In [ ]:
# C6. 啟動單元三 (最終合併自動駕駛)
robot_f.stop()
try:
    camera_f.unobserve(execute_final, names='value')
except Exception:
    pass
angle_last_f = 0.0
control_time_last_f = None
pd_initialized_f = False
stop_cooldown_f = 0.0
rail_cooldown_f = 0.0
stop_until_f = 0.0
pedestrian_until_f = 0.0
blocked_latched_f = False
safety_latched_f = False
sign_counts_f = {0: 0, 1: 0, 2: 0, 3: 0}

frame_deadline = time.monotonic() + 3.0
while camera_f.value is None and time.monotonic() < frame_deadline:
    time.sleep(0.05)
if camera_f.value is None:
    raise RuntimeError("單元三相機在 3 秒內未提供影格，馬達維持停止")
execute_final({'new': camera_f.value})
if not safety_latched_f and not blocked_latched_f:
    camera_f.observe(execute_final, names='value')
elif safety_latched_f:
    raise RuntimeError("單元三首次推論失敗，已保持馬達停止")
print("🚗 單元三：雙模型合併自動駕駛啟動！")

In [ ]:
# C7. 安全停止單元三
import gc
try:
    camera_f.unobserve(execute_final, names='value')
except Exception as e:
    print(f"Unobserve camera_f failed: {e}")
time.sleep(1.5)
robot_f.stop()
pd_initialized_f = False
control_time_last_f = None
camera_stopped_f = False
try:
    camera_f.stop()
    camera_stopped_f = True
except Exception as e:
    print(f"Stop camera_f failed: {e}")

try:
    cuda.Context.synchronize()
except Exception as e:
    print(f"CUDA synchronize failed: {e}")
# 只有底層相機成功停止後才清除 singleton
if camera_stopped_f and hasattr(Camera, '_instance') and Camera._instance is camera_f:
    Camera._instance = None
globals().pop('model_road_final', None)
globals().pop('model_sign_final', None)
globals().pop('cuda_context_f', None)
globals().pop('camera_f', None)
gc.collect()
torch.cuda.empty_cache()
if camera_stopped_f:
    print("單元三已停止，雙模型、相機與 GPU cache 已釋放 ✓")
else:
    print('⚠️ 警告：相機未能成功關閉，建議手動關閉或重啟 Kernel。')
